In [40]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-30B-A3B", trust_remote_code=True)

In [29]:
score_files = glob.glob('data/routing_scores/*.parquet')
dfs = pd.concat([pd.read_parquet(f) for f in score_files], ignore_index=True)

In [ ]:
dft = pd.read_parquet('data/request_text/all_request_text_data.parquet')
dft['token_str'] = tokenizer.batch_decode(dft['token_actual_id'], clean_up_tokenization_spaces=True)

In [30]:
dfs['request_id'].unique()

array([202,  92, 216,  97, 103, 143, 238, 211, 249, 100, 120, 115, 220,
        18, 133,  43, 102, 229, 221,  90,  81,  79, 233, 240, 227, 118,
        35, 114, 163, 171,  41,  30, 203, 157,   1, 182, 242, 247,  51,
        95, 113,  76,  87, 250, 106, 141,   3, 209, 251, 191,  39, 111,
       173, 181,  89,  74,  84, 190, 195, 162, 125, 214,  69, 132, 147,
        88,   2,  52, 196,  63, 252,  37, 156, 201,  83, 194, 123,  13,
       142,  72, 121,  53,  34,  19, 134, 176, 145, 207, 161, 178, 152,
       155, 204, 150, 188, 127, 236, 137,  58,  16,  75,  26,  40, 144,
       219, 255, 212,  64, 193, 166, 153, 140,  22,  61, 149, 222,  14,
        67, 186,  23,  31,  93, 179,   0,  99, 108,  11,  28,  48, 129,
       192,   8,  85, 169, 136,   6,  21, 124, 223, 244,   4, 174, 170,
       245, 154, 246, 217, 119, 228, 187,  78,  56,  77, 168,  57,  66,
       205, 159, 130,  25, 122, 180,  47,  55,  62, 158, 112, 104, 109,
        20, 139, 215, 101, 224, 131, 235, 167,  70,  96, 218, 16

In [22]:
dft.columns

Index(['request_id', 'token_position_in_sequence', 'token_actual_id',
       'token_str'],
      dtype='object')

In [23]:
dfs.columns

Index(['request_id', 'token_position_in_sequence', 'layer_id', 'expert_id',
       'routing_score'],
      dtype='object')

In [ ]:
tids = tokenizer("hello").input_ids
assert len(tids) == 1
tid = tids[0]

prim_keys = ['request_id', 'token_position_in_sequence']

matched = dft[dft['token_actual_id'] == tid][prim_keys]

token_scores = matched.merge(dfs, on=prim_keys, how='left')

token_scores

,request_id,token_position_in_sequence,layer_id,expert_id,routing_score
0,43,3,0,0,0.007111
1,43,3,0,1,0.002701
2,43,3,0,2,0.018188
3,43,3,0,3,0.010071
4,43,3,0,4,0.006287
...,...,...,...,...,...
36859,210,806,47,123,0.000115
36860,210,806,47,124,0.006073
36861,210,806,47,125,0.009094
36862,210,806,47,126,0.007568


In [43]:
matched

,request_id,token_position_in_sequence
32708,43,3
54537,96,3
91819,187,3
95303,194,3
95309,194,9
101063,210,806


In [111]:
req_to_plot = 210
token_to_plot = 41

# df_filtered = token_scores[(token_scores['request_id'] == req_to_plot) & (token_scores['token_position_in_sequence'] == token_to_plot)]

df_filtered = dfs[(dfs['request_id'] == req_to_plot) & (dfs['token_position_in_sequence'] == token_to_plot)]

if df_filtered.empty:
    print(f"错误：在DataFrame中找不到 token_position_in_sequence = {token_to_plot} 的数据。")
else:
    # 3. 创建数据透视表 (pivot table)
    # 横向是 expert_id，纵向是 layer_id，值为 routing_score
    # 我们需要处理可能存在的重复项 (layer_id, expert_id 对)，这里我们取平均值
    heatmap_data = pd.pivot_table(
        df_filtered,
        values='routing_score',
        index='layer_id',
        columns='expert_id',
        aggfunc='mean' # 如果每个(layer_id, expert_id)只有一个routing_score，用 'first' 或 'mean' 都可以
    )

    if heatmap_data.empty:
        print(f"错误：无法为 token_position_in_sequence = {token_to_plot} 创建热力图数据。检查筛选条件和数据。")
    else:
        # 4. 使用 seaborn 绘制热力图 (seaborn 基于 matplotlib，更易用)
        plt.figure(figsize=(100, 60)) #调整图像大小
        ax = sns.heatmap(heatmap_data, annot=True, fmt=".4f", cmap="viridis", linewidths=.5)

        # 5. 设置图表标题和坐标轴标签
        plt.title(f'Routing Score Heatmap for Req {req_to_plot} Token {token_to_plot}')
        plt.xlabel('Expert ID')
        plt.ylabel('Layer ID')

        # 6. 高亮 Top-K 专家
        def mark_pos(target_layer, target_expert):
            if target_layer in heatmap_data.index and target_expert in heatmap_data.columns:
                row_idx_highlight = heatmap_data.index.get_loc(target_layer)
                col_idx_highlight = heatmap_data.columns.get_loc(target_expert)
                ax.add_patch(plt.Rectangle((col_idx_highlight, row_idx_highlight), 1, 1,
                                        fill=False, edgecolor='red', lw=3))
        
        scores_data = heatmap_data.to_numpy()
        top_k = 8
        experts_for_layer = scores_data.argpartition(-top_k, axis=1)[:, -top_k:]
        for layer in heatmap_data.index:
            for expert in range(top_k):
                expert_id = experts_for_layer[layer][expert].item()
                mark_pos(layer, expert_id)

        # 7. 显示图表
        plt.tight_layout() # 自动调整布局，防止标签重叠
        plt.savefig('data/images/substrings-substring.png')
        plt.show()

In [104]:
print(tokenizer.decode(dft[dft['request_id'] == 210]['token_actual_id']))

<|im_start|>user
solve this task:

You are given a 0-indexed string s and a dictionary of words dictionary. You have to break s into one or more non-overlapping substrings such that each substring is present in dictionary. There may be some extra characters in s which are not present in any of the substrings.

Return the minimum number of extra characters left over if you break up s optimally.

Input: s = "leetscode", dictionary = ["leet","code","leetcode"]
Output: 1
Explanation: We can break s in two substrings: "leet" from index 0 to 3 and "code" from index 5 to 8. There is only 1 unused character (at index 4), so we return 1.<|im_end|>
<|im_start|>assistant
One solution to this problem is to use a sliding window approach. We can maintain a window of characters from the string s that we are currently processing, and for each character in the window, we can check if it is in the dictionary. If it is not, we add it to a list of extra characters. At the end, the length of this list is t

In [106]:
dft[dft['request_id'] == 210][0:50]

,request_id,token_position_in_sequence,token_actual_id,token_str
100257,210,0,151644,<|im_start|>
100258,210,1,872,user
100259,210,2,198,\n
100260,210,3,59419,solve
100261,210,4,419,this
100262,210,5,3383,task
100263,210,6,1447,:\n\n
100264,210,7,2610,You
100265,210,8,525,are
100266,210,9,2661,given


In [81]:
tokenizer.encode('你好')

[108386]